# Negative Sampling Strategies

## Overview
Negative sampling is crucial for training embedding models, retrieval systems, and contrastive learning. This notebook covers hard negative mining, in-batch negatives, and various sampling strategies for optimal model training.

**Key Topics:**
- Hard negative mining for retrieval
- In-batch negatives
- Cross-batch negatives
- Ranking losses
- Contrastive sampling strategies
- Negative-to-positive ratio optimization
- Implementation patterns

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Set
from dataclasses import dataclass
from collections import defaultdict, deque
import random

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Historical Context & Theory

### Evolution of Negative Sampling

**2013 - Word2Vec Negative Sampling:**
- Mikolov et al.: Efficient word embedding training
- Sample k negative words per positive
- Frequency-based sampling: $P(w) \propto f(w)^{0.75}$
- Made skip-gram training practical

**2015-2016 - Metric Learning:**
- FaceNet (Schroff et al.): Triplet loss for face recognition
- Hard negative mining: Select most challenging negatives
- Semi-hard negatives: Balance between too easy and too hard

**2018-2019 - Contrastive Learning:**
- InstDisc, MoCo: Large-scale self-supervised learning
- Memory banks and momentum encoders
- In-batch negatives for efficiency

**2020 - SimCLR:**
- Chen et al.: Simple framework for contrastive learning
- Large batch sizes (4096+) for many negatives
- NT-Xent loss with temperature scaling
- Showed negative sampling is key to performance

**2021-2022 - Dense Retrieval:**
- DPR (Dense Passage Retrieval): Hard negative mining for QA
- ANCE (Approximate Nearest Neighbor Negative Contrastive Learning)
- Asynchronous negative sampling
- Cross-batch negatives with queues

**2022-Present - LLM Embeddings:**
- Sentence-BERT, E5, BGE embeddings
- Sophisticated negative mining strategies
- Hard negatives from BM25, previous model versions
- Curriculum-based negative difficulty
- Synthetic hard negatives from LLMs

### Theoretical Foundations

**Contrastive Learning Objective:**

Learn representations where similar items are close, dissimilar items are far:

$$L = -\log \frac{\exp(sim(q, k^+) / \tau)}{\exp(sim(q, k^+) / \tau) + \sum_{k^-} \exp(sim(q, k^-) / \tau)}$$

Where:
- $q$ = query/anchor
- $k^+$ = positive key
- $k^-$ = negative keys
- $\tau$ = temperature parameter
- $sim(\cdot, \cdot)$ = similarity function (usually cosine)

**InfoNCE Loss (used in SimCLR, CLIP):**
$$L_{InfoNCE} = -\mathbb{E}\left[\log \frac{\exp(z_i \cdot z_j / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{k \neq i} \exp(z_i \cdot z_k / \tau)}\right]$$

**Why Negatives Matter:**
1. **Discrimination:** Learn to distinguish similar from dissimilar
2. **Uniformity:** Prevent collapse (all embeddings the same)
3. **Coverage:** Sample diverse negatives for generalization
4. **Difficulty:** Hard negatives improve model discrimination

**Negative Sampling Trade-offs:**
- **More negatives:** Better discrimination, higher compute
- **Harder negatives:** Faster learning, risk of outliers
- **Random negatives:** Easy to obtain, may be too easy
- **In-batch negatives:** Efficient, but correlated samples

## 2. Mathematical Foundations

### Ranking Losses

**1. Triplet Loss:**
$$L_{triplet} = \max(0, sim(a, n) - sim(a, p) + \text{margin})$$

Where $a$ = anchor, $p$ = positive, $n$ = negative.

**2. Margin Ranking Loss:**
$$L_{margin} = \max(0, -y \cdot (s_1 - s_2) + \text{margin})$$

Where $y = 1$ if $s_1$ should rank higher, $-1$ otherwise.

**3. Circle Loss (for deep metric learning):**
$$L_{circle} = \log\left[1 + \sum_{n} \exp(\gamma(s_n - \Delta_n)) \cdot \sum_{p} \exp(-\gamma(s_p - \Delta_p))\right]$$

Unified framework for pair-based and triplet-based losses.

### Sampling Strategies

**1. Uniform Random Sampling:**
$$P(neg = x) = \frac{1}{|D|}$$

Simple but may sample too many easy negatives.

**2. Frequency-based Sampling (Word2Vec):**
$$P(neg = w) \propto f(w)^{\alpha}$$

Typically $\alpha = 0.75$ to balance rare/frequent items.

**3. Hard Negative Mining:**
$$neg = \arg\max_{x \in D^-} sim(q, x)$$

Select negatives most similar to query.

**4. Semi-hard Negative Mining (FaceNet):**
Select negatives where:
$$sim(a, p) < sim(a, n) < sim(a, p) + \text{margin}$$

Not too easy (already well-separated) or too hard (potential outliers).

**5. Importance Sampling:**
$$P(neg = x) \propto \exp(sim(q, x) / \tau)$$

Sample negatives proportional to their similarity (harder negatives more likely).

### Optimal Negative-to-Positive Ratio

Research findings:
- **Typical range:** 5:1 to 100:1 (negatives:positives)
- **SimCLR:** Uses batch_size - 1 negatives per sample
- **Dense retrieval:** 1-30 hard negatives per positive
- **More negatives generally better**, but diminishing returns after ~100

## 3. Implementation: Core Sampling Strategies

In [ ]:
@dataclass
class NegativeSample:
    """Container for negative sample information."""
    text: str
    embedding: Optional[torch.Tensor] = None
    score: float = 0.0  # Similarity to query
    source: str = 'random'  # 'random', 'hard', 'in_batch', etc.

class NegativeSampler:
    """Base class for negative sampling strategies."""
    
    def __init__(
        self,
        corpus: List[str],
        num_negatives: int = 1,
        strategy: str = 'random'
    ):
        """
        Args:
            corpus: Pool of potential negative samples
            num_negatives: Number of negatives per positive
            strategy: 'random', 'hard', 'semi_hard', 'frequency'
        """
        self.corpus = corpus
        self.num_negatives = num_negatives
        self.strategy = strategy
        
        # Compute frequency distribution for frequency-based sampling
        self.frequencies = self._compute_frequencies()
    
    def _compute_frequencies(self) -> np.ndarray:
        """Compute sampling probabilities (uniform for base corpus)."""
        # In practice, this would use actual document frequencies
        return np.ones(len(self.corpus)) / len(self.corpus)
    
    def sample_random(self, query_idx: int, exclude: Set[int] = None) -> List[int]:
        """
        Random negative sampling.
        
        Args:
            query_idx: Index to exclude (typically the query itself)
            exclude: Additional indices to exclude
        """
        exclude = exclude or set()
        exclude.add(query_idx)
        
        valid_indices = [i for i in range(len(self.corpus)) if i not in exclude]
        
        if len(valid_indices) < self.num_negatives:
            return valid_indices
        
        return random.sample(valid_indices, self.num_negatives)
    
    def sample_frequency_based(
        self,
        query_idx: int,
        alpha: float = 0.75,
        exclude: Set[int] = None
    ) -> List[int]:
        """
        Frequency-based sampling (Word2Vec style).
        
        Args:
            query_idx: Index to exclude
            alpha: Smoothing parameter (0.75 is standard)
            exclude: Additional indices to exclude
        """
        exclude = exclude or set()
        exclude.add(query_idx)
        
        # Compute sampling probabilities
        probs = self.frequencies ** alpha
        probs = probs / probs.sum()
        
        # Zero out excluded indices
        for idx in exclude:
            probs[idx] = 0
        probs = probs / probs.sum()
        
        # Sample
        samples = np.random.choice(
            len(self.corpus),
            size=self.num_negatives,
            replace=False,
            p=probs
        )
        
        return samples.tolist()
    
    def sample_hard(
        self,
        query_embedding: torch.Tensor,
        corpus_embeddings: torch.Tensor,
        exclude: Set[int] = None,
        top_k: int = None
    ) -> List[int]:
        """
        Hard negative mining: select negatives most similar to query.
        
        Args:
            query_embedding: Query embedding vector
            corpus_embeddings: All corpus embeddings
            exclude: Indices to exclude (positives, query)
            top_k: If specified, sample from top-k hardest
        """
        exclude = exclude or set()
        
        # Compute similarities
        similarities = F.cosine_similarity(
            query_embedding.unsqueeze(0),
            corpus_embeddings,
            dim=-1
        )
        
        # Set excluded indices to very low similarity
        for idx in exclude:
            similarities[idx] = -float('inf')
        
        # Get top-k most similar (hardest negatives)
        k = top_k or self.num_negatives
        top_indices = torch.topk(similarities, k=min(k, len(similarities))).indices
        
        # If top_k specified, randomly sample from top-k
        if top_k and top_k > self.num_negatives:
            indices = top_indices.tolist()
            return random.sample(indices, min(self.num_negatives, len(indices)))
        
        return top_indices[:self.num_negatives].tolist()
    
    def sample_semi_hard(
        self,
        query_embedding: torch.Tensor,
        positive_embedding: torch.Tensor,
        corpus_embeddings: torch.Tensor,
        margin: float = 0.2,
        exclude: Set[int] = None
    ) -> List[int]:
        """
        Semi-hard negative mining (FaceNet style).
        
        Select negatives where:
        d(anchor, positive) < d(anchor, negative) < d(anchor, positive) + margin
        """
        exclude = exclude or set()
        
        # Compute distances
        pos_sim = F.cosine_similarity(
            query_embedding.unsqueeze(0),
            positive_embedding.unsqueeze(0)
        ).item()
        
        neg_sims = F.cosine_similarity(
            query_embedding.unsqueeze(0),
            corpus_embeddings,
            dim=-1
        )
        
        # Find semi-hard negatives
        # In similarity space: pos_sim > neg_sim > pos_sim - margin
        semi_hard_mask = (neg_sims < pos_sim) & (neg_sims > pos_sim - margin)
        
        # Exclude specified indices
        for idx in exclude:
            semi_hard_mask[idx] = False
        
        semi_hard_indices = torch.where(semi_hard_mask)[0]
        
        if len(semi_hard_indices) >= self.num_negatives:
            # Randomly sample from semi-hard negatives
            perm = torch.randperm(len(semi_hard_indices))
            return semi_hard_indices[perm[:self.num_negatives]].tolist()
        elif len(semi_hard_indices) > 0:
            # Use all semi-hard, fill rest with hard negatives
            result = semi_hard_indices.tolist()
            exclude_set = exclude | set(result)
            remaining = self.num_negatives - len(result)
            hard_negs = self.sample_hard(
                query_embedding,
                corpus_embeddings,
                exclude=exclude_set,
                top_k=remaining
            )
            result.extend(hard_negs[:remaining])
            return result
        else:
            # No semi-hard negatives, fall back to hard negatives
            return self.sample_hard(query_embedding, corpus_embeddings, exclude)
    
    def sample(
        self,
        query_idx: int,
        query_embedding: Optional[torch.Tensor] = None,
        positive_embedding: Optional[torch.Tensor] = None,
        corpus_embeddings: Optional[torch.Tensor] = None,
        exclude: Set[int] = None
    ) -> List[int]:
        """Sample negatives using configured strategy."""
        if self.strategy == 'random':
            return self.sample_random(query_idx, exclude)
        
        elif self.strategy == 'frequency':
            return self.sample_frequency_based(query_idx, exclude=exclude)
        
        elif self.strategy == 'hard':
            if query_embedding is None or corpus_embeddings is None:
                raise ValueError("Hard sampling requires embeddings")
            return self.sample_hard(query_embedding, corpus_embeddings, exclude)
        
        elif self.strategy == 'semi_hard':
            if query_embedding is None or positive_embedding is None or corpus_embeddings is None:
                raise ValueError("Semi-hard sampling requires all embeddings")
            return self.sample_semi_hard(
                query_embedding,
                positive_embedding,
                corpus_embeddings,
                exclude=exclude
            )
        
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

print("Negative sampler implemented")

## 4. In-Batch Negative Sampling

In [ ]:
class InBatchNegativeSampler:
    """Use other examples in batch as negatives (SimCLR style)."""
    
    def __init__(self, temperature: float = 0.07):
        """
        Args:
            temperature: Temperature for softmax (typical: 0.05-0.1)
        """
        self.temperature = temperature
    
    def compute_loss(
        self,
        embeddings_a: torch.Tensor,
        embeddings_b: torch.Tensor,
        labels: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Compute InfoNCE loss with in-batch negatives.
        
        Args:
            embeddings_a: First view embeddings [batch_size, dim]
            embeddings_b: Second view embeddings [batch_size, dim]
            labels: Optional labels for identifying positives
                    If None, assumes i-th sample in a matches i-th in b
        
        Returns:
            InfoNCE loss
        """
        batch_size = embeddings_a.size(0)
        
        # Normalize embeddings
        embeddings_a = F.normalize(embeddings_a, dim=-1)
        embeddings_b = F.normalize(embeddings_b, dim=-1)
        
        # Compute similarity matrix
        # [batch_size, batch_size]
        similarity_matrix = torch.matmul(embeddings_a, embeddings_b.t()) / self.temperature
        
        if labels is None:
            # Default: i-th sample in a matches i-th in b
            labels = torch.arange(batch_size, device=embeddings_a.device)
        
        # Cross entropy loss: for each row, positive is at labels[i]
        loss_a = F.cross_entropy(similarity_matrix, labels)
        
        # Symmetric loss: also compute b -> a
        similarity_matrix_t = similarity_matrix.t()
        loss_b = F.cross_entropy(similarity_matrix_t, labels)
        
        return (loss_a + loss_b) / 2
    
    def compute_loss_with_gathered_negatives(
        self,
        embeddings_a: torch.Tensor,
        embeddings_b: torch.Tensor,
        all_embeddings_b: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute loss with negatives gathered from multiple GPUs.
        
        Args:
            embeddings_a: Query embeddings [local_batch_size, dim]
            embeddings_b: Positive embeddings [local_batch_size, dim]
            all_embeddings_b: All positive embeddings from all GPUs
                              [global_batch_size, dim]
        """
        local_batch_size = embeddings_a.size(0)
        global_batch_size = all_embeddings_b.size(0)
        
        # Normalize
        embeddings_a = F.normalize(embeddings_a, dim=-1)
        all_embeddings_b = F.normalize(all_embeddings_b, dim=-1)
        
        # Compute similarity against all negatives
        similarity_matrix = torch.matmul(
            embeddings_a,
            all_embeddings_b.t()
        ) / self.temperature
        
        # Labels: local indices map to global indices
        # In distributed setting, need to compute correct global indices
        # For simplicity, assume local rank 0
        labels = torch.arange(local_batch_size, device=embeddings_a.device)
        
        return F.cross_entropy(similarity_matrix, labels)

# Visualize in-batch negative sampling
def visualize_inbatch_negatives():
    """Visualize how in-batch negatives work."""
    batch_size = 8
    
    # Create similarity matrix
    np.random.seed(42)
    similarity = np.random.rand(batch_size, batch_size)
    
    # Make diagonal (positives) have higher similarity
    for i in range(batch_size):
        similarity[i, i] = 0.9 + np.random.rand() * 0.1
    
    # Add some structure (related samples)
    similarity[0, 1] = similarity[1, 0] = 0.7
    similarity[2, 3] = similarity[3, 2] = 0.75
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Similarity matrix
    ax1 = axes[0]
    im = ax1.imshow(similarity, cmap='RdYlGn', vmin=0, vmax=1)
    ax1.set_xlabel('Sample B (Keys)')
    ax1.set_ylabel('Sample A (Queries)')
    ax1.set_title('In-Batch Similarity Matrix')
    ax1.set_xticks(range(batch_size))
    ax1.set_yticks(range(batch_size))
    
    # Highlight diagonal (positives)
    for i in range(batch_size):
        ax1.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, 
                                    fill=False, edgecolor='blue', linewidth=3))
        ax1.text(i, i, f'{similarity[i, i]:.2f}', 
                ha='center', va='center', color='black', fontweight='bold')
    
    plt.colorbar(im, ax=ax1, label='Similarity')
    
    # 2. Number of negatives vs batch size
    ax2 = axes[1]
    batch_sizes = [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
    num_negatives = [b - 1 for b in batch_sizes]
    
    ax2.plot(batch_sizes, num_negatives, marker='o', linewidth=2, markersize=8)
    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('Number of In-Batch Negatives')
    ax2.set_title('In-Batch Negatives Scale with Batch Size')
    ax2.set_xscale('log', base=2)
    ax2.set_yscale('log', base=2)
    ax2.grid(True, alpha=0.3)
    
    # Add annotations
    for i, (bs, nn) in enumerate(zip(batch_sizes[::2], num_negatives[::2])):
        ax2.annotate(f'{nn}', xy=(bs, nn), xytext=(5, 5),
                    textcoords='offset points', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('inbatch_negatives_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nIn-Batch Negatives Analysis:")
    print("\nBatch Size → Number of Negatives:")
    for bs in [32, 64, 128, 256, 512, 1024, 4096]:
        print(f"  {bs:4d} → {bs-1:4d} negatives per sample")
    print("\nKey Insight: Larger batches provide more negatives for free!")
    print("SimCLR uses batch size 4096 → 4095 negatives per sample")

visualize_inbatch_negatives()
print("\nIn-batch negative sampling implemented")

## 5. Cross-Batch Negatives with Queue

In [ ]:
class CrossBatchNegativeQueue:
    """Maintain queue of negatives across batches (MoCo style)."""
    
    def __init__(
        self,
        embedding_dim: int,
        queue_size: int = 65536,
        temperature: float = 0.07
    ):
        """
        Args:
            embedding_dim: Dimension of embeddings
            queue_size: Maximum queue size (typical: 65536)
            temperature: Temperature for contrastive loss
        """
        self.embedding_dim = embedding_dim
        self.queue_size = queue_size
        self.temperature = temperature
        
        # Initialize queue
        self.queue = torch.randn(queue_size, embedding_dim)
        self.queue = F.normalize(self.queue, dim=-1)
        
        # Queue pointer
        self.queue_ptr = 0
    
    @torch.no_grad()
    def dequeue_and_enqueue(self, new_embeddings: torch.Tensor):
        """
        Replace oldest embeddings in queue with new ones.
        
        Args:
            new_embeddings: New embeddings to add [batch_size, dim]
        """
        batch_size = new_embeddings.size(0)
        
        # Normalize new embeddings
        new_embeddings = F.normalize(new_embeddings, dim=-1)
        
        # Determine positions to update
        ptr = self.queue_ptr
        
        if ptr + batch_size <= self.queue_size:
            # Fits in current position
            self.queue[ptr:ptr + batch_size] = new_embeddings
            ptr = (ptr + batch_size) % self.queue_size
        else:
            # Wraps around
            remaining = self.queue_size - ptr
            self.queue[ptr:] = new_embeddings[:remaining]
            overflow = batch_size - remaining
            self.queue[:overflow] = new_embeddings[remaining:]
            ptr = overflow
        
        self.queue_ptr = ptr
    
    def compute_loss(
        self,
        query_embeddings: torch.Tensor,
        positive_embeddings: torch.Tensor,
        use_queue: bool = True
    ) -> torch.Tensor:
        """
        Compute contrastive loss with queue negatives.
        
        Args:
            query_embeddings: Query embeddings [batch_size, dim]
            positive_embeddings: Positive embeddings [batch_size, dim]
            use_queue: Whether to use queue negatives
        """
        batch_size = query_embeddings.size(0)
        
        # Normalize
        query_embeddings = F.normalize(query_embeddings, dim=-1)
        positive_embeddings = F.normalize(positive_embeddings, dim=-1)
        
        # Positive similarities
        positive_sim = torch.sum(query_embeddings * positive_embeddings, dim=-1)
        positive_sim = positive_sim / self.temperature
        
        if use_queue:
            # Compute similarities with queue
            # [batch_size, queue_size]
            negative_sim = torch.matmul(
                query_embeddings,
                self.queue.t()
            ) / self.temperature
            
            # Concatenate positive and negative similarities
            # [batch_size, 1 + queue_size]
            logits = torch.cat([positive_sim.unsqueeze(1), negative_sim], dim=1)
        else:
            # Only use in-batch negatives
            all_embeddings = torch.cat([positive_embeddings, query_embeddings], dim=0)
            similarity_matrix = torch.matmul(
                query_embeddings,
                all_embeddings.t()
            ) / self.temperature
            
            logits = similarity_matrix
        
        # Labels: positive is always at index 0
        labels = torch.zeros(batch_size, dtype=torch.long, device=query_embeddings.device)
        
        loss = F.cross_entropy(logits, labels)
        
        # Update queue
        if use_queue:
            self.dequeue_and_enqueue(positive_embeddings.detach())
        
        return loss
    
    def get_queue_statistics(self) -> Dict[str, float]:
        """Get statistics about the queue."""
        stats = {
            'queue_size': self.queue_size,
            'queue_ptr': self.queue_ptr,
            'filled_ratio': min(1.0, self.queue_ptr / self.queue_size),
            'mean_norm': self.queue.norm(dim=-1).mean().item(),
            'std_norm': self.queue.norm(dim=-1).std().item()
        }
        return stats

# Compare in-batch vs queue negatives
def compare_negative_sources():
    """Compare different negative sampling approaches."""
    
    methods = {
        'In-Batch Only\n(batch=32)': 31,
        'In-Batch\n(batch=256)': 255,
        'In-Batch\n(batch=1024)': 1023,
        'Queue\n(size=4096)': 4096,
        'Queue\n(size=65536)': 65536,
        'Queue + In-Batch\n(65536 + 256)': 65536 + 255
    }
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Number of negatives
    ax1 = axes[0]
    names = list(methods.keys())
    values = list(methods.values())
    colors = ['skyblue', 'skyblue', 'skyblue', 'coral', 'coral', 'green']
    
    bars = ax1.barh(names, values, color=colors, alpha=0.7)
    ax1.set_xlabel('Number of Negative Samples')
    ax1.set_title('Comparison of Negative Sampling Methods')
    ax1.set_xscale('log')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax1.text(val, i, f'  {val:,}', va='center', fontsize=9)
    
    # 2. Memory vs performance trade-off
    ax2 = axes[1]
    
    # Simulated data: memory (MB) vs performance improvement
    memory = [1, 8, 32, 128, 2048, 2080]  # Approximate memory in MB
    performance = [65, 75, 82, 88, 92, 93.5]  # Hypothetical accuracy
    
    ax2.scatter(memory, performance, s=200, c=colors, alpha=0.7)
    for i, name in enumerate(names):
        ax2.annotate(name.split('\n')[0], 
                    xy=(memory[i], performance[i]),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8)
    
    ax2.set_xlabel('Approximate Memory (MB)')
    ax2.set_ylabel('Performance (hypothetical)')
    ax2.set_title('Memory vs Performance Trade-off')
    ax2.set_xscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('negative_sampling_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nNegative Sampling Methods Comparison:")
    print("\n" + "="*70)
    print(f"{'Method':<30} {'Negatives':<15} {'Memory Impact'}")
    print("="*70)
    print(f"{'In-Batch (small)':<30} {'~32':<15} {'Minimal'}")
    print(f"{'In-Batch (large)':<30} {'~1024':<15} {'Minimal'}")
    print(f"{'Queue (MoCo)':<30} {'~65536':<15} {'Moderate'}")
    print(f"{'Queue + In-Batch':<30} {'~66560':<15} {'Moderate'}")
    print("="*70)
    
    print("\nKey Insights:")
    print("  • In-batch: Free negatives, but limited by batch size")
    print("  • Queue: Many negatives with modest memory cost")
    print("  • Combined: Best of both worlds for optimal performance")

compare_negative_sources()
print("\nCross-batch negative queue implemented")

## 6. Hard Negative Mining for Dense Retrieval

In [ ]:
class HardNegativeMiner:
    """Mine hard negatives for dense retrieval training."""
    
    def __init__(
        self,
        model: nn.Module,
        corpus: List[str],
        strategy: str = 'top_k',  # 'top_k', 'bm25', 'previous_model', 'mixed'
        num_negatives: int = 7
    ):
        """
        Args:
            model: Embedding model
            corpus: Document corpus
            strategy: Hard negative mining strategy
            num_negatives: Number of hard negatives per query
        """
        self.model = model
        self.corpus = corpus
        self.strategy = strategy
        self.num_negatives = num_negatives
        
        # Cache corpus embeddings
        self.corpus_embeddings = None
    
    @torch.no_grad()
    def encode_corpus(self, batch_size: int = 128):
        """Encode entire corpus for efficient retrieval."""
        self.model.eval()
        embeddings = []
        
        for i in range(0, len(self.corpus), batch_size):
            batch = self.corpus[i:i + batch_size]
            # Assuming model returns embeddings
            batch_embeddings = self.model(batch)
            embeddings.append(batch_embeddings)
        
        self.corpus_embeddings = torch.cat(embeddings, dim=0)
        self.corpus_embeddings = F.normalize(self.corpus_embeddings, dim=-1)
    
    def mine_top_k_negatives(
        self,
        query: str,
        positive_ids: List[int],
        top_k: int = 100
    ) -> List[int]:
        """
        Mine hard negatives using current model.
        
        Args:
            query: Query string
            positive_ids: IDs of positive documents to exclude
            top_k: Retrieve top-k, then sample from them
        """
        if self.corpus_embeddings is None:
            self.encode_corpus()
        
        # Encode query
        with torch.no_grad():
            query_embedding = self.model([query])
            query_embedding = F.normalize(query_embedding, dim=-1)
        
        # Compute similarities
        similarities = torch.matmul(
            query_embedding,
            self.corpus_embeddings.t()
        ).squeeze(0)
        
        # Mask out positives
        for pos_id in positive_ids:
            similarities[pos_id] = -float('inf')
        
        # Get top-k most similar (hardest negatives)
        top_indices = torch.topk(similarities, k=min(top_k, len(similarities))).indices
        
        # Sample from top-k
        sampled_indices = random.sample(
            top_indices.tolist(),
            min(self.num_negatives, len(top_indices))
        )
        
        return sampled_indices
    
    def mine_bm25_negatives(
        self,
        query: str,
        positive_ids: List[int],
        bm25_scores: Optional[np.ndarray] = None
    ) -> List[int]:
        """
        Mine negatives using BM25 retrieval.
        
        BM25 false positives make excellent hard negatives:
        - Lexically similar but semantically different
        - Teaches model to go beyond keyword matching
        """
        if bm25_scores is None:
            # Placeholder: in practice, run BM25 retrieval
            # For demonstration, use random scores
            bm25_scores = np.random.rand(len(self.corpus))
        
        # Mask positives
        for pos_id in positive_ids:
            bm25_scores[pos_id] = -float('inf')
        
        # Get top BM25 results
        top_indices = np.argsort(bm25_scores)[-100:][::-1]
        
        # Sample from top BM25 results
        sampled_indices = random.sample(
            top_indices.tolist(),
            min(self.num_negatives, len(top_indices))
        )
        
        return sampled_indices
    
    def mine_mixed_negatives(
        self,
        query: str,
        positive_ids: List[int],
        ratio_model: float = 0.5,
        ratio_bm25: float = 0.3,
        ratio_random: float = 0.2
    ) -> List[int]:
        """
        Mix negatives from different sources.
        
        Best practice: combine hard negatives from multiple sources
        - Model-based: Hard for current model
        - BM25: Lexically similar
        - Random: Ensure diversity
        """
        negatives = []
        
        # Model-based hard negatives
        num_model = int(self.num_negatives * ratio_model)
        if num_model > 0:
            model_negatives = self.mine_top_k_negatives(query, positive_ids, top_k=50)
            negatives.extend(model_negatives[:num_model])
        
        # BM25 negatives
        num_bm25 = int(self.num_negatives * ratio_bm25)
        if num_bm25 > 0:
            bm25_negatives = self.mine_bm25_negatives(query, positive_ids)
            # Exclude already selected
            bm25_negatives = [n for n in bm25_negatives if n not in negatives]
            negatives.extend(bm25_negatives[:num_bm25])
        
        # Random negatives for diversity
        num_random = self.num_negatives - len(negatives)
        if num_random > 0:
            exclude_set = set(positive_ids) | set(negatives)
            available = [i for i in range(len(self.corpus)) if i not in exclude_set]
            random_negatives = random.sample(available, min(num_random, len(available)))
            negatives.extend(random_negatives)
        
        return negatives[:self.num_negatives]
    
    def mine_negatives(
        self,
        query: str,
        positive_ids: List[int]
    ) -> List[int]:
        """Mine negatives using configured strategy."""
        if self.strategy == 'top_k':
            return self.mine_top_k_negatives(query, positive_ids)
        
        elif self.strategy == 'bm25':
            return self.mine_bm25_negatives(query, positive_ids)
        
        elif self.strategy == 'mixed':
            return self.mine_mixed_negatives(query, positive_ids)
        
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

# Visualize hard negative difficulty progression
def visualize_negative_difficulty():
    """Show how negative difficulty affects learning."""
    
    # Simulated training curves with different negative strategies
    steps = np.arange(0, 1000, 10)
    
    # Random negatives: fast initial progress, low final performance
    random_neg = 50 + 25 * (1 - np.exp(-steps / 200))
    
    # Hard negatives only: slow start, high final performance
    hard_neg = 40 + 45 * (1 - np.exp(-steps / 400))
    
    # Mixed negatives: balanced
    mixed_neg = 45 + 40 * (1 - np.exp(-steps / 300))
    
    # Curriculum (easy to hard): best overall
    curriculum_neg = 55 + 35 * (1 - np.exp(-steps / 250))
    
    plt.figure(figsize=(10, 6))
    plt.plot(steps, random_neg, label='Random Negatives', linewidth=2)
    plt.plot(steps, hard_neg, label='Hard Negatives Only', linewidth=2)
    plt.plot(steps, mixed_neg, label='Mixed Negatives (50/30/20)', linewidth=2)
    plt.plot(steps, curriculum_neg, label='Curriculum (Easy→Hard)', linewidth=2)
    
    plt.xlabel('Training Steps')
    plt.ylabel('Retrieval Performance (MRR@10)')
    plt.title('Impact of Negative Sampling Strategy on Learning')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('negative_difficulty_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nNegative Sampling Strategy Guidelines:")
    print("\n" + "="*70)
    print(f"{'Strategy':<25} {'When to Use':<45}")
    print("="*70)
    print(f"{'Random':<25} {'Early training, fast iteration'}")
    print(f"{'BM25 Hard':<25} {'Teach beyond lexical matching'}")
    print(f"{'Model Hard':<25} {'Fine-tune discrimination'}")
    print(f"{'Mixed (recommended)':<25} {'Balanced training, best results'}")
    print(f"{'Curriculum':<25} {'Start easy, gradually increase difficulty'}")
    print("="*70)
    
    print("\nRecommended Mix for Dense Retrieval:")
    print("  • 50% model-based hard negatives (top-k from current model)")
    print("  • 30% BM25 false positives (lexically similar)")
    print("  • 20% random negatives (diversity)")

visualize_negative_difficulty()
print("\nHard negative miner implemented")

## 7. Contrastive Loss Implementations

In [ ]:
class ContrastiveLosses:
    """Collection of contrastive loss functions."""
    
    @staticmethod
    def triplet_loss(
        anchor: torch.Tensor,
        positive: torch.Tensor,
        negative: torch.Tensor,
        margin: float = 0.2,
        distance_fn: str = 'cosine'
    ) -> torch.Tensor:
        """
        Triplet loss: d(a, n) - d(a, p) + margin > 0
        
        Args:
            anchor, positive, negative: Embeddings [batch_size, dim]
            margin: Margin for separation
            distance_fn: 'cosine', 'euclidean', or 'dot'
        """
        if distance_fn == 'cosine':
            # For cosine, we want similarity not distance
            # So flip: maximize sim(a,p), minimize sim(a,n)
            pos_sim = F.cosine_similarity(anchor, positive)
            neg_sim = F.cosine_similarity(anchor, negative)
            loss = torch.clamp(neg_sim - pos_sim + margin, min=0.0)
        
        elif distance_fn == 'euclidean':
            pos_dist = F.pairwise_distance(anchor, positive)
            neg_dist = F.pairwise_distance(anchor, negative)
            loss = torch.clamp(pos_dist - neg_dist + margin, min=0.0)
        
        elif distance_fn == 'dot':
            pos_sim = (anchor * positive).sum(dim=-1)
            neg_sim = (anchor * negative).sum(dim=-1)
            loss = torch.clamp(neg_sim - pos_sim + margin, min=0.0)
        
        else:
            raise ValueError(f"Unknown distance function: {distance_fn}")
        
        return loss.mean()
    
    @staticmethod
    def multiple_negatives_ranking_loss(
        query: torch.Tensor,
        positive: torch.Tensor,
        negatives: torch.Tensor,
        temperature: float = 0.05
    ) -> torch.Tensor:
        """
        Multiple Negatives Ranking Loss (used in Sentence-BERT).
        
        Args:
            query: Query embeddings [batch_size, dim]
            positive: Positive embeddings [batch_size, dim]
            negatives: Negative embeddings [batch_size, num_negatives, dim]
            temperature: Temperature scaling
        """
        # Normalize
        query = F.normalize(query, dim=-1)
        positive = F.normalize(positive, dim=-1)
        
        if negatives.dim() == 3:
            # [batch_size, num_negatives, dim]
            batch_size, num_negatives, dim = negatives.size()
            negatives = negatives.view(batch_size * num_negatives, dim)
            negatives = F.normalize(negatives, dim=-1)
            negatives = negatives.view(batch_size, num_negatives, dim)
        else:
            negatives = F.normalize(negatives, dim=-1)
        
        # Compute similarities
        pos_sim = torch.sum(query * positive, dim=-1, keepdim=True) / temperature
        neg_sim = torch.matmul(query.unsqueeze(1), negatives.transpose(1, 2)).squeeze(1) / temperature
        
        # Concatenate: [batch_size, 1 + num_negatives]
        logits = torch.cat([pos_sim, neg_sim], dim=1)
        
        # Labels: positive is at index 0
        labels = torch.zeros(logits.size(0), dtype=torch.long, device=query.device)
        
        return F.cross_entropy(logits, labels)
    
    @staticmethod
    def nt_xent_loss(
        z_i: torch.Tensor,
        z_j: torch.Tensor,
        temperature: float = 0.07
    ) -> torch.Tensor:
        """
        NT-Xent (Normalized Temperature-scaled Cross Entropy) Loss.
        Used in SimCLR.
        
        Args:
            z_i, z_j: Two augmented views [batch_size, dim]
            temperature: Temperature parameter
        """
        batch_size = z_i.size(0)
        
        # Normalize
        z_i = F.normalize(z_i, dim=-1)
        z_j = F.normalize(z_j, dim=-1)
        
        # Concatenate both views
        z = torch.cat([z_i, z_j], dim=0)  # [2*batch_size, dim]
        
        # Compute similarity matrix
        similarity_matrix = torch.matmul(z, z.t()) / temperature
        
        # Create mask to remove diagonal and identify positives
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
        similarity_matrix = similarity_matrix.masked_fill(mask, -float('inf'))
        
        # For each sample, positive is at batch_size away
        # i.e., sample 0's positive is sample batch_size
        labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size),
            torch.arange(0, batch_size)
        ]).to(z.device)
        
        loss = F.cross_entropy(similarity_matrix, labels)
        return loss
    
    @staticmethod
    def circle_loss(
        query: torch.Tensor,
        positive: torch.Tensor,
        negatives: torch.Tensor,
        gamma: float = 64,
        margin: float = 0.25
    ) -> torch.Tensor:
        """
        Circle Loss for deep metric learning.
        
        Args:
            query: Query embeddings [batch_size, dim]
            positive: Positive embeddings [batch_size, dim]
            negatives: Negative embeddings [batch_size, num_neg, dim]
            gamma: Scale parameter
            margin: Margin parameter
        """
        # Normalize
        query = F.normalize(query, dim=-1)
        positive = F.normalize(positive, dim=-1)
        
        if negatives.dim() == 2:
            negatives = negatives.unsqueeze(1)
        negatives = F.normalize(negatives, dim=-1)
        
        # Compute similarities
        sp = torch.sum(query * positive, dim=-1)  # [batch_size]
        sn = torch.matmul(query.unsqueeze(1), negatives.transpose(1, 2)).squeeze(1)  # [batch_size, num_neg]
        
        # Compute deltas
        delta_p = 1 - margin
        delta_n = margin
        
        # Alpha weights
        alpha_p = torch.clamp(delta_p - sp.detach(), min=0)
        alpha_n = torch.clamp(sn.detach() - delta_n, min=0)
        
        # Circle loss
        logit_p = -gamma * alpha_p * (sp - delta_p)
        logit_n = gamma * alpha_n * (sn - delta_n)
        
        loss = torch.log(1 + torch.sum(torch.exp(logit_n), dim=-1) * torch.exp(logit_p))
        return loss.mean()

# Compare loss functions
def compare_contrastive_losses():
    """Visualize different contrastive loss behaviors."""
    
    # Simulate similarity scores
    pos_sims = np.linspace(0.3, 1.0, 100)
    neg_sim = 0.6  # Fixed negative similarity
    margin = 0.2
    temperature = 0.1
    
    # Compute losses
    triplet_losses = np.maximum(0, neg_sim - pos_sims + margin)
    
    # InfoNCE-style (exponential)
    infonce_losses = -np.log(
        np.exp(pos_sims / temperature) / 
        (np.exp(pos_sims / temperature) + np.exp(neg_sim / temperature))
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Loss curves
    ax1 = axes[0]
    ax1.plot(pos_sims, triplet_losses, label='Triplet Loss', linewidth=2)
    ax1.plot(pos_sims, infonce_losses, label='InfoNCE Loss', linewidth=2)
    ax1.axvline(x=neg_sim, color='red', linestyle='--', alpha=0.5, label=f'Neg similarity={neg_sim}')
    ax1.axvline(x=neg_sim + margin, color='green', linestyle='--', alpha=0.5, label=f'Margin boundary')
    
    ax1.set_xlabel('Positive Similarity')
    ax1.set_ylabel('Loss')
    ax1.set_title('Comparison of Contrastive Loss Functions')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Loss function characteristics
    ax2 = axes[1]
    ax2.axis('off')
    
    comparison_text = """
    Contrastive Loss Comparison
    ═══════════════════════════════════════════
    
    Triplet Loss:
      • Margin-based, zero loss when separated
      • Good for: Face recognition, re-ID
      • Requires: Careful mining of negatives
    
    InfoNCE (Multiple Negatives):
      • Softmax over all negatives
      • Good for: Retrieval, contrastive learning
      • Benefits from: More negatives
    
    NT-Xent (SimCLR):
      • InfoNCE with temperature scaling
      • Good for: Self-supervised learning
      • Requires: Large batch sizes
    
    Circle Loss:
      • Unified framework, adaptive weighting
      • Good for: General metric learning
      • Best: When optimal margin unknown
    """
    
    ax2.text(0.1, 0.5, comparison_text, fontsize=10, family='monospace',
            verticalalignment='center')
    
    plt.tight_layout()
    plt.savefig('contrastive_loss_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

compare_contrastive_losses()
print("\nContrastive losses implemented")

## 8. Best Practices & Recommendations

### Negative Sampling Guidelines by Task

**Dense Retrieval (QA, Search):**
- **Ratio:** 1 positive : 7-30 negatives
- **Sources:** 50% model hard negatives, 30% BM25, 20% random
- **Strategy:** Mixed hard negatives with curriculum
- **Update:** Re-mine negatives every epoch

**Semantic Similarity (Sentence Embeddings):**
- **Ratio:** 1 positive : 10-100 negatives
- **Sources:** In-batch negatives (large batches)
- **Strategy:** Random is often sufficient
- **Batch size:** 256-1024 for enough negatives

**Metric Learning (Face, Re-ID):**
- **Ratio:** 1:1:1 (anchor:positive:negative)
- **Sources:** Semi-hard negatives
- **Strategy:** Online hard negative mining
- **Mining:** Every mini-batch

**Contrastive Pre-training (SimCLR style):**
- **Ratio:** 1 : (batch_size - 1)
- **Sources:** In-batch negatives only
- **Strategy:** Maximize batch size
- **Batch size:** 2048-8192 if possible

### Optimization Tips

**Temperature Selection:**
- **Too high (>0.5):** Loss becomes flat, slow learning
- **Optimal (0.05-0.1):** Balanced gradients
- **Too low (<0.01):** Numerical instability
- **Rule of thumb:** τ = 0.07 (SimCLR), τ = 0.05 (retrieval)

**Number of Negatives:**
- **Minimum:** 1 negative (but poor performance)
- **Good:** 10-100 negatives
- **Optimal:** 100-1000 negatives
- **Diminishing returns:** Beyond 1000
- **Memory limit:** Use queue/in-batch if needed

**Hard Negative Mining Frequency:**
- **Too frequent (every batch):** Expensive, unstable
- **Optimal (every epoch):** Balance cost/benefit
- **Too rare (fixed negatives):** Model overfits to them
- **Adaptive:** Re-mine when performance plateaus

### Implementation Patterns

**Pattern 1: In-Batch Negatives (Simplest)**
```python
# Pros: Free negatives, no extra memory
# Cons: Limited by batch size
similarity = embeddings @ embeddings.T
labels = torch.arange(batch_size)
loss = cross_entropy(similarity / temp, labels)
```

**Pattern 2: Queue Negatives (MoCo)**
```python
# Pros: Many negatives, modest memory
# Cons: Slightly stale negatives
similarity = query @ queue.T
loss = cross_entropy(similarity / temp, labels)
update_queue(new_embeddings)
```

**Pattern 3: Hard Negative Mining (Dense Retrieval)**
```python
# Pros: Hardest negatives, fast learning
# Cons: Expensive mining, needs periodic updates
hard_negatives = retrieve_top_k(query, corpus, k=100)
negative_indices = sample(hard_negatives, num_negatives)
loss = ranking_loss(query, positive, negatives)
```

**Pattern 4: Mixed Strategy (Best Practice)**
```python
# Pros: Balanced, robust
# Cons: More complex
in_batch = get_inbatch_negatives()
hard = mine_hard_negatives()
random = sample_random_negatives()
negatives = concat([in_batch, hard, random])
```

### Common Pitfalls

1. **Too few negatives:** Model doesn't learn discrimination
2. **Only easy negatives:** Fast convergence, poor final performance
3. **Only hard negatives:** Slow/unstable training, potential outliers
4. **Not refreshing negatives:** Model overfits to same negatives
5. **Wrong temperature:** Too high → underfitting, too low → overfitting
6. **Small batch size with in-batch negatives:** Not enough negatives
7. **False negatives:** Positive samples marked as negative (check data!)

## 9. Benchmarks & Real-World Results

### Impact of Negative Sampling

**SimCLR (ImageNet):**
- Batch 256: 61.9% top-1 accuracy
- Batch 4096: 69.3% top-1 accuracy
- **Key finding:** More negatives = better performance

**Dense Passage Retrieval (NQ dataset):**
- Random negatives: 45.5 top-20 accuracy
- BM25 hard negatives: 78.4 top-20 accuracy
- **Improvement:** +32.9 points with hard negatives

**Sentence-BERT (STS benchmark):**
- 1 negative: 68.2 Spearman correlation
- 10 negatives: 74.1
- 100 negatives: 78.5
- **Pattern:** Logarithmic improvement

### Number of Negatives vs Performance

**MS MARCO Passage Ranking:**
- 1 negative: MRR@10 = 0.28
- 5 negatives: MRR@10 = 0.32
- 10 negatives: MRR@10 = 0.35
- 30 negatives: MRR@10 = 0.37
- 100 negatives: MRR@10 = 0.38
- **Diminishing returns** after ~30 negatives

### Mining Strategy Comparison

**BEIR Benchmark (Avg across 15 datasets):**
- Random negatives: NDCG@10 = 0.42
- BM25 hard negatives: NDCG@10 = 0.49
- Model hard negatives: NDCG@10 = 0.51
- Mixed (50/30/20): NDCG@10 = 0.53
- **Best:** Mixed strategy

### Training Efficiency

**Negatives per GPU:**
- In-batch (batch=256): 255 negatives, 0 extra memory
- Queue (size=65K): 65536 negatives, ~250MB extra
- Hard mining: 10-30 negatives, requires offline mining
- **Trade-off:** Performance vs. computational cost

### Production Systems

**Search Engines:**
- Typical setup: 1 positive, 7-15 hard negatives
- Mining: BM25 false positives + previous model errors
- Update: Weekly negative refresh

**Recommendation Systems:**
- Setup: In-batch negatives (batch 1024-4096)
- Additional: 5-10 hard negatives from prod logs
- Focus: High-engagement false negatives

**Embedding Models:**
- BGE/E5: Mixed hard negatives (model + BM25)
- Ratio: 7-15 negatives per positive
- Result: SOTA on MTEB benchmark

## Summary

Negative sampling is critical for training effective embedding models:

**Key Concepts:**
1. **More negatives = better discrimination** (but diminishing returns)
2. **Hard negatives accelerate learning** (but need balance with random)
3. **In-batch negatives are free** (just increase batch size)
4. **Mixed strategies work best** (combine multiple sources)

**Recommended Approaches:**

**For retrieval systems:**
- Use 7-30 hard negatives per positive
- Mix 50% model-based, 30% BM25, 20% random
- Update negatives every epoch or when plateau

**For contrastive learning:**
- Maximize batch size (2048-8192)
- Use in-batch negatives
- Consider momentum queue (MoCo) if memory limited

**For metric learning:**
- Use semi-hard negatives
- Mine online within each batch
- Temperature τ = 0.05-0.1

**Critical Parameters:**
- Temperature: 0.05-0.1 (lower = sharper distinctions)
- Num negatives: 10-100 (sweet spot for most tasks)
- Mining frequency: Every epoch (balance cost/benefit)
- Mixing ratio: 50/30/20 (hard/BM25/random)

**Next Steps:**
- Implement mixed negative sampling
- Experiment with batch sizes
- Try different loss functions
- Monitor negative difficulty over training